# TC Tracking Runner - E3SM Hindcast (ne30pg2)

This notebook only configures and calls `scripts/run_tc_track_e3sm.py` to process TC tracks with TempestExtremes.

**Pipeline:** `DetectNodes` -> `StitchNodes` -> `HistogramNodes`
**Output layout:** `OUTDIR / CASE / MEMBER / post / atm / tc-analysis /`

Run this notebook first to generate the stitched track text files and histogram NetCDF files used by the diagnostics notebook.


In [1]:
import os
import sys
import subprocess
import xarray as xr
import matplotlib.pyplot as plt

# PROJ data path — auto-resolved when running under the e3sm_analysis kernel
# (CONDA_PREFIX is set).  Hardcoded fallback only needed if kernel is not activated.
_proj_path = os.path.join(os.environ.get("CONDA_PREFIX", ""), "share", "proj")
if not os.path.exists(_proj_path):
    _proj_path = "/global/homes/z/zhan391/.conda/envs/e3sm_analysis/share/proj"
os.environ["PROJ_LIB"]  = _proj_path
os.environ["PROJ_DATA"] = _proj_path

import cartopy.crs as ccrs

# Python interpreter for the tracking script.
# When the kernel IS e3sm_analysis, sys.executable is already correct.
PYTHON = sys.executable
print(f"PYTHON      : {PYTHON}")
print(f"CONDA_PREFIX: {os.environ.get('CONDA_PREFIX', '(not set — fallback paths used)')}")


PYTHON      : /global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python
CONDA_PREFIX: /global/common/software/e3sm/anaconda_envs/base


## Configuration — fill in ALL fields before running

Fields left as `None` will cause the **Validate** cell to raise an error.

In [2]:
import numpy as np
from pathlib import Path
from esp_lab import data_access_e3sm as data_access

# ------------------------------------------------------------------ #
#  PATHS  (all required)
# ------------------------------------------------------------------ #

SIM_DIR      = Path("/global/cfs/cdirs/e3smdata/simulations/S2S2D")
OUTDIR       = Path("/global/cfs/cdirs/e3sm/S2S2D/post_process")
CONNECT_FILE = Path("/global/cfs/cdirs/e3sm/zhan391/TempestExtremes/grid_info/outCS_ne30pg2_connect.txt")
SCRIPT       = Path("/global/homes/z/zhan391/code/ESP-Lab/scripts/run_tc_track_e3sm.py")

# ------------------------------------------------------------------ #
#  CASES — built from prefix + init tags (same pattern as skill maps)
# ------------------------------------------------------------------ #

case_prefix = "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL"

years  = 1980
yeare  = 2018
yexcl  = None          # set to a year int to exclude, or None

init_months = [5, 11]  # May and November starts

lead_years = [y for y in np.arange(years, yeare + 1) if y != yexcl]

CASES = []
for init_month in init_months:
    init_tags = data_access.build_init_tags(lead_years, init_month)
    CASES += [f"{case_prefix}_{tag}" for tag in init_tags]

print(f"Total cases: {len(CASES)}")
print("First few:", CASES[:3])
print("Last few: ", CASES[-3:])

# ------------------------------------------------------------------ #
#  ENSEMBLE  (required)
# ------------------------------------------------------------------ #

case_nens = 10
members   = [f"EN{i:02d}" for i in range(case_nens)]
MEMBERS   = members
NENS      = None   # e.g. 3 for a quick test

# ------------------------------------------------------------------ #
#  GRID / STREAMS / PARSET
# ------------------------------------------------------------------ #

GRID            = "ne30pg2"
STREAM_TAG      = "eam.h2"
PHIS_STREAM_TAG = "eam.h0"
# Yeager-style warm-core setup: set3 = _DIFF(Z300,Z500), wc_mag=-6.0
# in scripts/run_tc_track_e3sm.py.
PARSET          = "set3"
WORKERS         = 1

# ------------------------------------------------------------------ #
#  TRACKING THRESHOLDS
#  Defaults below match the Yeager SMYLE driver in
#  notebooks/YeagerEA_GMD2022_submission/drive-tracking-SMYLE_feb.sh.
# ------------------------------------------------------------------ #

PSL_FO_MAG      = 200.0
PSL_FO_DIST     = 8.0
WC_FO_DIST      = 8.0
WC_MAX_OFFSET   = 3.0
MERGE_DIST      = 6.0
TRAJ_RANGE      = 10.0
TRAJ_MIN_LENGTH = "10"
TRAJ_MAX_GAP    = "3"
MAX_TOPO        = 150.0
MAX_LAT         = 50.0
MIN_WIND        = 8.0
SCI_DIST        = 9

# ------------------------------------------------------------------ #
#  TEMPESTEXTREMES + NCO bin directory
#  Auto-resolved from CONDA_PREFIX when running under e3sm_analysis kernel.
#  Hardcoded fallback for any other kernel.
# ------------------------------------------------------------------ #

_ENV_BIN = os.path.join(os.environ.get("CONDA_PREFIX", ""), "bin")
if not os.path.exists(os.path.join(_ENV_BIN, "DetectNodes")):
    _ENV_BIN = "/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin"

TE_BIN  = _ENV_BIN   # DetectNodes / StitchNodes / HistogramNodes
NCO_BIN = _ENV_BIN   # ncks / ncwa

print(f"_ENV_BIN : {_ENV_BIN}")


Total cases: 78
First few: ['WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1981050100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1982050100']
Last few:  ['WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_2016110100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_2017110100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_2018110100']
_ENV_BIN : /global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin


## Validate — run this before anything else

In [3]:
import subprocess, xarray as xr, matplotlib.pyplot as plt, cartopy.crs as ccrs

# ---- enforce all required fields are set ----
_required = {
    "SIM_DIR":         SIM_DIR,
    "CASES":           CASES,
    "OUTDIR":          OUTDIR,
    "CONNECT_FILE":    CONNECT_FILE,
    "SCRIPT":          SCRIPT,
    "MEMBERS":         MEMBERS,
    "GRID":            GRID,
    "STREAM_TAG":      STREAM_TAG,
    "PHIS_STREAM_TAG": PHIS_STREAM_TAG,
    "TE_BIN":          TE_BIN,
    "PARSET":          PARSET,
    "WORKERS":         WORKERS,
    "PSL_FO_MAG":      PSL_FO_MAG,
    "PSL_FO_DIST":     PSL_FO_DIST,
    "WC_FO_DIST":      WC_FO_DIST,
    "WC_MAX_OFFSET":   WC_MAX_OFFSET,
    "MERGE_DIST":      MERGE_DIST,
    "TRAJ_RANGE":      TRAJ_RANGE,
    "TRAJ_MIN_LENGTH": TRAJ_MIN_LENGTH,
    "TRAJ_MAX_GAP":    TRAJ_MAX_GAP,
    "MAX_TOPO":        MAX_TOPO,
    "MAX_LAT":         MAX_LAT,
    "MIN_WIND":        MIN_WIND,
    "SCI_DIST":        SCI_DIST,
}
_missing = [k for k, v in _required.items() if v is None]
if _missing:
    raise ValueError(
        "The following required fields are still None — fill them in the Configuration cell:\n"
        + "\n".join(f"  {k}" for k in _missing)
    )

# ---- check each case directory exists ----
_errors = []
if not SIM_DIR.is_dir():
    _errors.append(f"SIM_DIR does not exist: {SIM_DIR}")
for _case in CASES:
    _cdir = SIM_DIR / _case
    if not _cdir.is_dir():
        _errors.append(f"Case directory not found: {_cdir}")
if not SCRIPT.is_file():
    _errors.append(f"Tracking script not found: {SCRIPT}")
if _errors:
    raise FileNotFoundError("Path validation failed:\n" + "\n".join(_errors))

# ---- resolve member list from the first case ----
_ref_case_dir = SIM_DIR / CASES[0]
_all_members = sorted(p.name for p in _ref_case_dir.iterdir()
                      if p.is_dir() and p.name.startswith("EN"))
members_avail = list(MEMBERS) if MEMBERS else _all_members
if not members_avail:
    raise ValueError(f"No EN* member directories found in {_ref_case_dir}")
if NENS is not None:
    members_avail = members_avail[:NENS]

print("Configuration valid ✓")
print(f"  SIM_DIR       : {SIM_DIR}")
print(f"  Cases         : {len(CASES)} total  ({CASES[0]}  ...  {CASES[-1]})")
print(f"  OUTDIR        : {OUTDIR}")
print(f"  CONNECT_FILE  : {CONNECT_FILE}  {'✓ exists' if CONNECT_FILE.exists() else '(will auto-generate)'}")
print(f"  SCRIPT        : {SCRIPT}")
print(f"  Members       : {members_avail}  (NENS={NENS})")
print(f"  Grid          : {GRID}")
print(f"  Streams       : {STREAM_TAG} / {PHIS_STREAM_TAG}")
print(f"  Parset        : {PARSET}")
print(f"  Workers       : {WORKERS}")
print("  Tracking args : "
      f"psl={PSL_FO_MAG}/{PSL_FO_DIST}, "
      f"wc_dist={WC_FO_DIST}, wc_offset={WC_MAX_OFFSET}, "
      f"range={TRAJ_RANGE}, minlen={TRAJ_MIN_LENGTH}, maxgap={TRAJ_MAX_GAP}, "
      f"wind>={MIN_WIND}, sci_dist={SCI_DIST}, maxlat={MAX_LAT}, maxtopo={MAX_TOPO}")
print(f"\n  Example output: {OUTDIR / CASES[0] / members_avail[0] / 'post' / 'atm' / 'tc-analysis'}")


Configuration valid ✓
  SIM_DIR       : /global/cfs/cdirs/e3smdata/simulations/S2S2D
  Cases         : 78 total  (WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_2018110100)
  OUTDIR        : /global/cfs/cdirs/e3sm/S2S2D/post_process
  CONNECT_FILE  : /global/cfs/cdirs/e3sm/zhan391/TempestExtremes/grid_info/outCS_ne30pg2_connect.txt  ✓ exists
  SCRIPT        : /global/homes/z/zhan391/code/ESP-Lab/scripts/run_tc_track_e3sm.py
  Members       : ['EN00', 'EN01', 'EN02', 'EN03', 'EN04', 'EN05', 'EN06', 'EN07', 'EN08', 'EN09']  (NENS=None)
  Grid          : ne30pg2
  Streams       : eam.h2 / eam.h0
  Parset        : set3
  Workers       : 1
  Tracking args : psl=200.0/8.0, wc_dist=8.0, wc_offset=3.0, range=10.0, minlen=10, maxgap=3, wind>=8.0, sci_dist=9, maxlat=50.0, maxtopo=150.0

  Example output: /global/cfs/cdirs/e3sm/S2S2D/post_process/WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100/EN00/post/atm/tc-anal

## 1.  Survey available `eam.h2` files

In [4]:
# Survey h2/h0 file counts for first two cases (all members)
for case in CASES[:2]:
    case_dir = SIM_DIR / case
    print(f"\nCase : {case}")
    for member in members_avail:
        hist_dir = case_dir / member / "archive" / "atm" / "hist"
        h2_files = sorted(hist_dir.glob(f"*.{STREAM_TAG}.*.nc"))
        h0_files = sorted(hist_dir.glob(f"*.{PHIS_STREAM_TAG}.*.nc"))
        print(f"  {member}")
        print(f"    {STREAM_TAG}  : {len(h2_files)} files", end="")
        if h2_files:
            print(f"  [{h2_files[0].name}  ...  {h2_files[-1].name}]")
        else:
            print("  <none found>")
        print(f"    {PHIS_STREAM_TAG} : {len(h0_files)} files", end="")
        if h0_files:
            print(f"  [{h0_files[0].name}  ...  {h0_files[-1].name}]")
        else:
            print("  <none found>")



Case : WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100
  EN00
    eam.h2  : 25 files  [WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.eam.h2.1980-05-01-00000.nc  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.eam.h2.1982-04-21-00000.nc]
    eam.h0 : 24 files  [WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.eam.h0.1980-05.nc  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.eam.h0.1982-04.nc]
  EN01
    eam.h2  : 25 files  [WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN01.eam.h2.1980-05-01-00000.nc  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN01.eam.h2.1982-04-21-00000.nc]
    eam.h0 : 24 files  [WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN01.eam.h0.1980-05.nc  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN01.eam.h0.1982-04.nc]
  EN02
    eam.h2  : 25 files  [WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1

## 2.  Verify TC variables in `eam.h2`

Required variables are selected from `PARSET` (`set3` uses `PSL`, `UBOT`, `VBOT`, `Z300`, `Z500`).  
`PHIS` comes from a static file extracted from `eam.h0`.

In [5]:
# Verify TC variables in the first available case/member
TC_VARS_BY_PARSET = {
    "set1": {"PSL", "UBOT", "VBOT", "Z200", "Z500"},
    "set2": {"PSL", "UBOT", "VBOT", "T200", "T500"},
    "set3": {"PSL", "UBOT", "VBOT", "Z300", "Z500"},
    "set4": {"PSL", "UBOT", "VBOT", "T300", "T500"},
}
TC_VARS_REQUIRED = TC_VARS_BY_PARSET[PARSET]

case_dir = SIM_DIR / CASES[0]
member   = members_avail[0]
hist_dir = case_dir / member / "archive" / "atm" / "hist"

h2_files = sorted(hist_dir.glob(f"*.{STREAM_TAG}.*.nc"))
assert h2_files, f"No {STREAM_TAG} files in {hist_dir}"
ds_h2    = xr.open_dataset(h2_files[0], decode_times=False)
all_vars = set(ds_h2.data_vars) | set(ds_h2.coords)
found    = TC_VARS_REQUIRED & all_vars
missing  = TC_VARS_REQUIRED - all_vars

print(f"Case   : {CASES[0]}")
print(f"Member : {member}")
print(f"Stream : {STREAM_TAG}  ({h2_files[0].name})")
print(f"  dims   : {dict(ds_h2.sizes)}")
print(f"  found  : {sorted(found)}")
print(f"  missing: {sorted(missing) or 'none  ✓'}")

h0_files = sorted(hist_dir.glob(f"*.{PHIS_STREAM_TAG}.*.nc"))
assert h0_files, f"No {PHIS_STREAM_TAG} files in {hist_dir}"
ds_h0   = xr.open_dataset(h0_files[0], decode_times=False)
phis_ok = "PHIS" in ds_h0.data_vars or "PHIS" in ds_h0.coords
print(f"\nStream : {PHIS_STREAM_TAG}  ({h0_files[0].name})")
print(f"  PHIS  : {'present ✓' if phis_ok else 'MISSING  <-- problem'}")


Case   : WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100
Member : EN00
Stream : eam.h2  (WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.eam.h2.1980-05-01-00000.nc)
  dims   : {'ncol': 21600, 'lev': 80, 'ilev': 81, 'P3_input_dim': 16, 'P3_output_dim': 32, 'cosp_prs': 7, 'nbnd': 2, 'cosp_tau': 7, 'cosp_scol': 10, 'cosp_ht': 40, 'cosp_temp': 40, 'cosp_sr': 15, 'cosp_sza': 5, 'cosp_htmisr': 16, 'cosp_tau_modis': 7, 'cosp_reffice': 6, 'cosp_reffliq': 6, 'cosp_iwp_modis': 7, 'cosp_lwp_modis': 7, 'time': 120}
  found  : ['PSL', 'UBOT', 'VBOT', 'Z300', 'Z500']
  missing: none  ✓

Stream : eam.h0  (WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.eam.h0.1980-05.nc)
  PHIS  : present ✓


## 3.  Check connectivity file

TempestExtremes requires a connectivity file for the native ne30pg2 unstructured grid.  
If it is missing, the tracking script will auto-generate it via `--grid`.

In [6]:
if CONNECT_FILE.exists():
    size_mb = CONNECT_FILE.stat().st_size / 1e6
    print(f"Connect file : {CONNECT_FILE}")
    print(f"  Size : {size_mb:.1f} MB  ✓")
    with open(CONNECT_FILE) as f:
        for i, line in enumerate(f):
            print(f"  line {i+1}: {line.rstrip()}")
            if i >= 3:
                break
else:
    print(f"Connect file not found: {CONNECT_FILE}")
    print(f"It will be auto-generated via --grid {GRID} when the tracking script runs.")

Connect file : /global/cfs/cdirs/e3sm/zhan391/TempestExtremes/grid_info/outCS_ne30pg2_connect.txt
  Size : 1.9 MB  ✓
  line 1: #TempestGridConnectivityFileV2.0
  line 2: 1,21600
  line 3: 3.15740716043964e+02,-3.49114391634599e+01,5.18997015195786e-04,4,2,3,10918,17883
  line 4: 3.17240673319773e+02,-3.55819651497593e+01,5.27971397106035e-04,4,1,4,5,17884


## 4.  Dry-run — preview commands without executing

In [7]:
import time

def build_cmd(case: str, members: list, *, dry_run: bool, force: bool = False) -> list:
    """Build CLI command for a single case + member list."""
    cmd = [
        PYTHON, str(SCRIPT),
        "--sim-dir",         str(SIM_DIR),
        "--outdir",          str(OUTDIR),
        "--cases",           case,
        "--members",         *members,
        "--parset",          PARSET,
        "--stream-tag",      STREAM_TAG,
        "--phis-stream-tag", PHIS_STREAM_TAG,
        "--connect-file",    str(CONNECT_FILE),
        "--grid",            GRID,
        "--te-bin",          TE_BIN,
        "--workers",         str(WORKERS),
        "--psl-fo-mag",     str(PSL_FO_MAG),
        "--psl-fo-dist",    str(PSL_FO_DIST),
        "--wc-fo-dist",     str(WC_FO_DIST),
        "--wc-max-offset",  str(WC_MAX_OFFSET),
        "--merge-dist",     str(MERGE_DIST),
        "--traj-range",     str(TRAJ_RANGE),
        "--traj-min-length", str(TRAJ_MIN_LENGTH),
        "--traj-max-gap",   str(TRAJ_MAX_GAP),
        "--max-topo",       str(MAX_TOPO),
        "--max-lat",        str(MAX_LAT),
        "--min-wind",       str(MIN_WIND),
        "--sci-dist",       str(SCI_DIST),
        "--verbose",
    ]
    if NCO_BIN:
        cmd += ["--nco-bin", NCO_BIN]
    if NENS is not None:
        cmd += ["--nens", str(NENS)]
    if dry_run:
        cmd += ["--dry-run"]
    if force:
        cmd += ["--force"]
    return cmd


def run_cases(cases_to_run, members_to_run, *, dry_run: bool, force: bool = False):
    """Loop over cases, run the tracking script, print per-case progress."""
    tag = "DRY-RUN" if dry_run else "RUN"
    print(f"{tag}: {len(cases_to_run)} cases × {len(members_to_run)} members"
          + (f"  |  force={force}" if not dry_run else ""))
    print()

    results_log = {}
    t0_total = time.time()

    for i, case in enumerate(cases_to_run, 1):
        t0 = time.time()
        print(f"[{i:3d}/{len(cases_to_run)}]  {case}")

        cmd = build_cmd(case, members_to_run, dry_run=dry_run, force=force)
        result = subprocess.run(cmd, capture_output=True, text=True)

        elapsed = time.time() - t0
        status  = "✓" if result.returncode == 0 else "✗ FAILED"
        results_log[case] = result.returncode

        for line in result.stdout.splitlines():
            if any(kw in line for kw in ("Summary", "OK", "Skipped", "Failed", "No data", "Dry-run", "Would process")):
                print(f"         {line.strip()}")

        print(f"         → {status}  ({elapsed:.0f}s)\n")

        if result.returncode != 0:
            print("  === STDERR ===")
            print(result.stderr[-1000:])

    elapsed_total = time.time() - t0_total
    n_ok     = sum(1 for rc in results_log.values() if rc == 0)
    n_failed = sum(1 for rc in results_log.values() if rc != 0)
    print(f"\n{'='*60}")
    print(f"{tag}  {len(cases_to_run)} cases  |  ✓ {n_ok} OK  |  ✗ {n_failed} failed  |  {elapsed_total/60:.1f} min")
    if n_failed:
        print("Failed cases:")
        for case, rc in results_log.items():
            if rc != 0:
                print(f"  {case}  (exit {rc})")
    return results_log


# ---- Dry-run: same case/member selection as the real run ----
# Adjust these to test a subset before committing to the full run:
#   cases_to_run   = CASES[:3]
#   members_to_run = ["EN00", "EN01"]
cases_to_run   = CASES
members_to_run = members_avail

dry_run_log = run_cases(cases_to_run, members_to_run, dry_run=True)


DRY-RUN: 78 cases × 10 members

[  1/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100
         → ✓  (0s)

[  2/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1981050100
         → ✓  (0s)

[  3/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1982050100
         → ✓  (0s)

[  4/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1983050100
         → ✓  (0s)

[  5/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1984050100
         → ✓  (0s)

[  6/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1985050100
         → ✓  (0s)

[  7/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1986050100
         → ✓  (0s)

[  8/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1987050100
         → ✓  (0s)

[  9/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1988050100
         → ✓  (0s)

[ 10/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1989050100
         → ✓  (0s)

[ 11/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIR

## 5. Run TC Tracking

> **Prerequisites:**
> - `DetectNodes`, `StitchNodes`, `HistogramNodes` must be on `PATH` or available through `TE_BIN`.
> - Each member can take several minutes; use a batch job for large ensembles.
> - To re-run existing output, change `FORCE = False` to `FORCE = True`.


In [8]:
# ---- Production run ----
# Adjust to process a subset:
#   cases_to_run   = CASES[:3]
#   members_to_run = ["EN00", "EN01"]
cases_to_run   = CASES
members_to_run = members_avail

FORCE = False   # set True to overwrite existing output

run_log = run_cases(cases_to_run, members_to_run, dry_run=False, force=FORCE)


RUN: 78 cases × 10 members  |  force=False

[  1/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100
         → ✓  (0s)

[  2/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1981050100
         → ✓  (0s)

[  3/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1982050100
         → ✓  (0s)

[  4/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1983050100
         → ✓  (0s)

[  5/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1984050100
         → ✓  (0s)

[  6/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1985050100
         → ✓  (0s)

[  7/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1986050100
         → ✓  (0s)

[  8/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1987050100
         → ✓  (0s)

[  9/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1988050100
         → ✓  (0s)

[ 10/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1989050100
         → ✓  (0s)

[ 11/78]  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5